In [1]:
!pip install albumentations timm

In [2]:
import torch
from torch import nn
from torchvision import models as resnet_model
import torch.nn.functional as F
import torchvision
import sys
sys.path.append('/kaggle/input/models/anushka1511/res2net-v1b-py/pytorch/default/1/')
from res2net_v1b import res2net50_v1b_26w_4s
import numpy as np
from timm.models.layers import DropPath
from timm.models.layers import trunc_normal_
import math

__all__ = ['Pra_FATNet']

# ---- Detect device automatically ----
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")


class SpectralGatingNetwork(nn.Module):
    def __init__(self, dim, layer_id=None):
        super().__init__()
        if layer_id == 0:
            self.h = 24
            self.w = 13
        elif layer_id == 1:
            self.h = 48
            self.w = 25
        elif layer_id == 2:
            self.h = 96
            self.w = 49
        elif layer_id == 3:
            self.h = 12
            self.w = 7

        self.dim = dim

    def forward(self, x):
        B, C, H, W = x.shape

        h = H
        w = h // 2 + 1
        # Create weight on the same device as input
        complex_weight = nn.Parameter(
            torch.randn(self.dim, h, w, 2, dtype=torch.float32) * 0.02
        ).to(x.device)

        x = x.to(torch.float32)
        x = torch.fft.rfft2(x, dim=(2, 3), norm='ortho')

        weight = torch.view_as_complex(complex_weight)
        x = x * weight
        x = torch.fft.irfft2(x, s=(H, W), dim=(2, 3), norm='ortho')
        x = x.reshape(B, C, H, W)
        return x


class BasicConv2d(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size, stride=1, padding=0, dilation=1):
        super(BasicConv2d, self).__init__()
        self.conv = nn.Conv2d(in_planes, out_planes,
                              kernel_size=kernel_size, stride=stride,
                              padding=padding, dilation=dilation, bias=False)
        self.bn = nn.BatchNorm2d(out_planes)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return x


class RFB_modified(nn.Module):
    def __init__(self, in_channel, out_channel):
        super(RFB_modified, self).__init__()
        self.relu = nn.ReLU(True)
        self.branch0 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
        )
        self.branch1 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 3), padding=(0, 1)),
            BasicConv2d(out_channel, out_channel, kernel_size=(3, 1), padding=(1, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=3, dilation=3)
        )
        self.branch2 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 5), padding=(0, 2)),
            BasicConv2d(out_channel, out_channel, kernel_size=(5, 1), padding=(2, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=5, dilation=5)
        )
        self.branch3 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 7), padding=(0, 3)),
            BasicConv2d(out_channel, out_channel, kernel_size=(7, 1), padding=(3, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=7, dilation=7)
        )
        self.conv_cat = BasicConv2d(4 * out_channel, out_channel, 3, padding=1)
        self.conv_res = BasicConv2d(in_channel, out_channel, 1)

    def forward(self, x):
        x0 = self.branch0(x)
        x1 = self.branch1(x)
        x2 = self.branch2(x)
        x3 = self.branch3(x)
        x_cat = self.conv_cat(torch.cat((x0, x1, x2, x3), 1))
        x = self.relu(x_cat + self.conv_res(x))
        return x


class aggregation(nn.Module):
    def __init__(self, channel):
        super(aggregation, self).__init__()
        self.relu = nn.ReLU(True)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv_upsample1 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample2 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample3 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample4 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample5 = BasicConv2d(2 * channel, 2 * channel, 3, padding=1)
        self.conv_concat2 = BasicConv2d(2 * channel, 2 * channel, 3, padding=1)
        self.conv_concat3 = BasicConv2d(3 * channel, 3 * channel, 3, padding=1)
        self.conv4 = BasicConv2d(3 * channel, 3 * channel, 3, padding=1)
        self.conv5 = nn.Conv2d(3 * channel, 1, 1)

    def forward(self, x1, x2, x3):
        x1_1 = x1
        x2_1 = self.conv_upsample1(self.upsample(x1)) * x2
        x3_1 = self.conv_upsample2(self.upsample(self.upsample(x1))) \
               * self.conv_upsample3(self.upsample(x2)) * x3
        x2_2 = torch.cat((x2_1, self.conv_upsample4(self.upsample(x1_1))), 1)
        x2_2 = self.conv_concat2(x2_2)
        x3_2 = torch.cat((x3_1, self.conv_upsample5(self.upsample(x2_2))), 1)
        x3_2 = self.conv_concat3(x3_2)
        x = self.conv4(x3_2)
        x = self.conv5(x)
        return x


class SpatialAttentionBlock(nn.Module):
    def __init__(self, in_channels):
        super(SpatialAttentionBlock, self).__init__()
        self.query = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 8, kernel_size=(1, 3), padding=(0, 1)),
            nn.BatchNorm2d(in_channels // 8),
            nn.ReLU(inplace=True)
        )
        self.key = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 8, kernel_size=(3, 1), padding=(1, 0)),
            nn.BatchNorm2d(in_channels // 8),
            nn.ReLU(inplace=True)
        )
        self.value = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.gamma = nn.Parameter(torch.zeros(1))
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        B, C, H, W = x.size()
        proj_query = self.query(x).view(B, -1, W * H).permute(0, 2, 1)
        proj_key = self.key(x).view(B, -1, W * H)
        affinity = torch.matmul(proj_query, proj_key)
        affinity = self.softmax(affinity)
        proj_value = self.value(x).view(B, -1, H * W)
        weights = torch.matmul(proj_value, affinity.permute(0, 2, 1))
        weights = weights.view(B, C, H, W)
        out = self.gamma * weights + x
        return out


class ChannelAttentionBlock(nn.Module):
    def __init__(self, in_channels):
        super(ChannelAttentionBlock, self).__init__()
        self.gamma = nn.Parameter(torch.zeros(1))
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        B, C, H, W = x.size()
        proj_query = x.view(B, C, -1)
        proj_key = x.view(B, C, -1).permute(0, 2, 1)
        affinity = torch.matmul(proj_query, proj_key)
        affinity_new = torch.max(affinity, -1, keepdim=True)[0].expand_as(affinity) - affinity
        affinity_new = self.softmax(affinity_new)
        proj_value = x.view(B, C, -1)
        weights = torch.matmul(affinity_new, proj_value)
        weights = weights.view(B, C, H, W)
        out = self.gamma * weights + x
        return out


class AffinityAttention(nn.Module):
    def __init__(self, in_channels):
        super(AffinityAttention, self).__init__()
        self.sab = SpatialAttentionBlock(in_channels)
        self.cab = ChannelAttentionBlock(in_channels)

    def forward(self, x):
        sab = self.sab(x)
        cab = self.cab(x)
        out = sab + cab
        return out


class squeeze_excitation_block(nn.Module):
    def __init__(self, in_channels, ratio=8):
        super().__init__()
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // ratio),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // ratio, in_channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        batch_size, channel_size, _, _ = x.size()
        y = self.avgpool(x).view(batch_size, channel_size)
        y = self.fc(y).view(batch_size, channel_size, 1, 1)
        return x * y.expand_as(x)


class DecoderBottleneckLayer(nn.Module):
    def __init__(self, in_channels, n_filters, use_transpose=True):
        super(DecoderBottleneckLayer, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels // 4, 1)
        self.norm1 = nn.BatchNorm2d(in_channels // 4)
        self.relu1 = nn.ReLU(inplace=True)
        if use_transpose:
            self.up = nn.Sequential(
                nn.ConvTranspose2d(
                    in_channels // 4, in_channels // 4, 3, stride=2, padding=1, output_padding=1
                ),
                nn.BatchNorm2d(in_channels // 4),
                nn.ReLU(inplace=True)
            )
        else:
            self.up = nn.Upsample(scale_factor=2, align_corners=True, mode="bilinear")
        self.conv3 = nn.Conv2d(in_channels // 4, n_filters, 1)
        self.norm3 = nn.BatchNorm2d(n_filters)
        self.relu3 = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv1(x)
        x = self.norm1(x)
        x = self.relu1(x)
        x = self.up(x)
        x = self.conv3(x)
        x = self.norm3(x)
        x = self.relu3(x)
        return x


class Conv2D(nn.Module):
    def __init__(self, in_c, out_c, kernel_size=7, padding=3, dilation=1, bias=False, act=True, stride=2):
        super().__init__()
        self.act = act
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=kernel_size, stride=stride,
                      padding=padding, dilation=dilation, bias=bias),
            nn.BatchNorm2d(out_c)
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        if self.act:
            x = self.relu(x)
        return x


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, kernel_size=3, padding=1, dilation=1, bias=False, act=True, stride=1):
        super().__init__()
        self.act = act
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=kernel_size, stride=stride,
                      padding=padding, dilation=dilation, bias=bias),
            nn.BatchNorm2d(out_c)
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        if self.act:
            x = self.relu(x)
        return x


class FAMBlock(nn.Module):
    def __init__(self, channels, layer_id, h):
        super(FAMBlock, self).__init__()
        self.conv3 = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, padding=1)
        self.conv1 = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=1)
        self.relu3 = nn.ReLU(inplace=True)
        self.relu1 = nn.ReLU(inplace=True)
        drop_path = 0.
        self.norm = nn.LayerNorm(h)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.specter = SpectralGatingNetwork(dim=channels, layer_id=layer_id)

    def forward(self, x):
        x0 = x
        x0 = self.norm(x0)
        x0 = self.specter(x0)
        x0 = self.drop_path(x0)
        x = x + x0
        x3 = self.conv3(x)
        x3 = self.relu3(x3)
        x1 = self.conv1(x)
        x1 = self.relu1(x1)
        out = x3 + x1
        return out


class decoderAttention(nn.Module):
    def __init__(self, in_channels, layer_id):
        super(decoderAttention, self).__init__()
        kernel_size = 3
        n_div = 4
        self.dim_conv = in_channels // n_div
        self.dim_untouched = in_channels - self.dim_conv * 3
        self.group = 1
        self.conv1 = nn.Conv2d(self.dim_conv, self.dim_conv, kernel_size,
                               stride=1, padding=(kernel_size - 1) // 2, groups=self.group)
        self.conv2_1 = nn.Conv2d(self.dim_conv, self.dim_conv, (1, 5), stride=1, padding=(0, 2), groups=self.group)
        self.conv2_2 = nn.Conv2d(self.dim_conv, self.dim_conv, (5, 1), stride=1, padding=(2, 0), groups=self.group)
        self.conv3_1 = nn.Conv2d(self.dim_conv, self.dim_conv, (1, 7), stride=1, padding=(0, 3), groups=self.group)
        self.conv3_2 = nn.Conv2d(self.dim_conv, self.dim_conv, (7, 1), stride=1, padding=(3, 0), groups=self.group)
        self.conv = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.sgn = SpectralGatingNetwork(dim=self.dim_untouched, layer_id=layer_id)

    def forward(self, x):
        x1, x2, x3, x4 = torch.split(x, [self.dim_conv, self.dim_conv, self.dim_conv, self.dim_untouched], dim=1)
        x1 = self.conv1(x1)
        x2 = self.conv2_2(self.conv2_1(x2))
        x3 = self.conv3_2(self.conv3_1(x3))
        x4 = self.sgn(x4)
        x = torch.cat((x1, x2, x3, x4), dim=1)
        x = self.conv(x)
        return x


class Pra_FATNet(nn.Module):
    def __init__(self, num_classes=1, input_channels=3, deep_supervision=True):
        super(Pra_FATNet, self).__init__()
        channel = 32
        self.resnet = res2net50_v1b_26w_4s(pretrained=True)
        self.rfb2_1 = RFB_modified(512, channel)
        self.rfb3_1 = RFB_modified(1024, channel)
        self.rfb4_1 = RFB_modified(2048, channel)
        self.affinity_attention = AffinityAttention(in_channels=2048)
        self.firstconv = self.resnet.conv1
        self.firstbn = self.resnet.bn1
        self.firstrelu = self.resnet.relu

        transformer = torch.hub.load('facebookresearch/deit:main', 'deit_base_distilled_patch16_384', pretrained=True)
        self.patch_embed = transformer.patch_embed
        self.transformers = nn.ModuleList([transformer.blocks[i] for i in range(12)])
        self.conv_seq_img = nn.Conv2d(in_channels=3072, out_channels=2048, kernel_size=1, padding=0)
        self.se = squeeze_excitation_block(in_channels=4096)
        self.conv2d = nn.Conv2d(in_channels=4096, out_channels=2048, kernel_size=1, padding=0)
        self.cb = ConvBlock(512, 32)
        self.cb2 = ConvBlock(32, 1)

        self.FAMBlock1 = FAMBlock(channels=256, layer_id=2, h=96)
        self.FAMBlock2 = FAMBlock(channels=512, layer_id=1, h=48)
        self.FAMBlock3 = FAMBlock(channels=1024, layer_id=0, h=24)
        self.FAM1 = nn.ModuleList([self.FAMBlock1 for i in range(6)])
        self.FAM2 = nn.ModuleList([self.FAMBlock2 for i in range(4)])
        self.FAM3 = nn.ModuleList([self.FAMBlock3 for i in range(2)])

        filters = [256, 512, 1024, 2048]
        self.decoder4 = DecoderBottleneckLayer(filters[3], filters[2])
        self.decoder3 = DecoderBottleneckLayer(filters[2], filters[1])
        self.decoder2 = DecoderBottleneckLayer(filters[1], filters[0])
        self.decoder1 = DecoderBottleneckLayer(filters[0], filters[0])

        self.final_decoder1 = DecoderBottleneckLayer(256, 128)
        self.final_decoder2 = DecoderBottleneckLayer(128, 64)
        self.final_conv1 = nn.Conv2d(64, 32, 3, 1, 1)
        self.final_relu1 = nn.ReLU(inplace=True)
        self.final_conv2 = nn.Conv2d(32, 32, 3, padding=1)
        self.final_relu2 = nn.ReLU(inplace=True)
        self.final_conv3 = nn.Conv2d(32, 1, 3, padding=1)

        self.featuredecoderAttention = decoderAttention(in_channels=2048, layer_id=3)
        self.decoderAttention4 = decoderAttention(in_channels=1024, layer_id=0)
        self.decoderAttention3 = decoderAttention(in_channels=512, layer_id=1)

        self.agg1 = aggregation(channel)
        self.ra4_conv1 = BasicConv2d(2048, 256, kernel_size=1)
        self.ra4_conv2 = BasicConv2d(256, 256, kernel_size=5, padding=2)
        self.ra4_conv3 = BasicConv2d(256, 256, kernel_size=5, padding=2)
        self.ra4_conv4 = BasicConv2d(256, 256, kernel_size=5, padding=2)
        self.ra4_conv5 = BasicConv2d(256, 1, kernel_size=1)
        self.ra3_conv1 = BasicConv2d(1024, 64, kernel_size=1)
        self.ra3_conv2 = BasicConv2d(64, 64, kernel_size=3, padding=1)
        self.ra3_conv3 = BasicConv2d(64, 64, kernel_size=3, padding=1)
        self.ra3_conv4 = BasicConv2d(64, 1, kernel_size=3, padding=1)
        self.ra2_conv1 = BasicConv2d(512, 64, kernel_size=1)
        self.ra2_conv2 = BasicConv2d(64, 64, kernel_size=3, padding=1)
        self.ra2_conv3 = BasicConv2d(64, 64, kernel_size=3, padding=1)
        self.ra2_conv4 = BasicConv2d(64, 1, kernel_size=3, padding=1)

    def forward(self, x):
        b, c, h, w = x.shape
        y = x

        x = self.firstconv(x)
        x = self.firstbn(x)
        x = self.firstrelu(x)
        x = self.resnet.maxpool(x)
        print('x', x.shape)

        x1 = self.resnet.layer1(x)
        x2 = self.resnet.layer2(x1)
        x3 = self.resnet.layer3(x2)
        x4 = self.resnet.layer4(x3)

        print('x1', x1.shape)
        print('x2', x2.shape)
        print('x3', x3.shape)
        print('x4', x4.shape)

        x2_rfb = self.rfb2_1(x2)
        x3_rfb = self.rfb3_1(x3)
        x4_rfb = self.rfb4_1(x4)

        emb = self.patch_embed(y)
        for i in range(12):
            emb = self.transformers[i](emb)
        print('emb', emb.shape)

        feature_tf = emb.permute(0, 2, 1)
        feature_tf = feature_tf.view(b, 3072, 12, 12)
        feature_tf = self.conv_seq_img(feature_tf)
        feature_cat = torch.cat((x4, feature_tf), dim=1)
        feature_att = self.se(feature_cat)
        feature_out = self.conv2d(feature_att)

        for i in range(2):
            e3 = self.FAM3[i](x3)
        for i in range(4):
            e2 = self.FAM2[i](x2)
        for i in range(6):
            e1 = self.FAM1[i](x1)

        print('e3', e3.shape)
        print('e2', e2.shape)
        print('e1', e1.shape)

        feature_out_attention = self.affinity_attention(feature_out)
        feature_out = feature_out_attention + feature_out

        d4 = self.decoder4(feature_out) + e3
        d3 = self.decoder3(d4) + e2
        d2 = self.decoder2(d3) + e1

        print('feature_out', feature_out.shape)
        print('d4', d4.shape)
        print('d3', d3.shape)
        print('d2', d2.shape)

        feature_out = self.featuredecoderAttention(feature_out)
        d4 = self.decoderAttention4(d4)
        d3 = self.decoderAttention3(d3)

        feature_out = self.ra4_conv1(feature_out)
        feature_out = F.relu(self.ra4_conv2(feature_out))
        feature_out = F.relu(self.ra4_conv3(feature_out))
        feature_out = F.relu(self.ra4_conv4(feature_out))
        feature_out = self.ra4_conv5(feature_out)

        d4 = self.ra3_conv1(d4)
        d4 = F.relu(self.ra3_conv2(d4))
        d4 = F.relu(self.ra3_conv3(d4))
        d4 = self.ra3_conv4(d4)

        d3 = self.ra2_conv1(d3)
        d3 = F.relu(self.ra2_conv2(d3))
        d3 = F.relu(self.ra2_conv3(d3))
        d3 = self.ra2_conv4(d3)

        ra5_feat = self.agg1(x4_rfb, x3_rfb, x2_rfb)
        print('ra5_feat', ra5_feat.shape)
        lateral_map_5 = F.interpolate(ra5_feat, scale_factor=8, mode='bilinear')

        crop_4 = F.interpolate(ra5_feat, scale_factor=0.25, mode='bilinear')
        x = -1 * (torch.sigmoid(crop_4)) + 1
        x = x.expand(-1, 2048, -1, -1).mul(x4)
        x = self.ra4_conv1(x)
        x = F.relu(self.ra4_conv2(x))
        x = F.relu(self.ra4_conv3(x))
        x = F.relu(self.ra4_conv4(x))
        ra4_feat = self.ra4_conv5(x)
        print('ra4_feat', ra4_feat.shape)
        x = ra4_feat + crop_4 + feature_out
        lateral_map_4 = F.interpolate(x, scale_factor=32, mode='bilinear')

        crop_3 = F.interpolate(x, scale_factor=2, mode='bilinear')
        x = -1 * (torch.sigmoid(crop_3)) + 1
        x = x.expand(-1, 1024, -1, -1).mul(x3)
        x = self.ra3_conv1(x)
        x = F.relu(self.ra3_conv2(x))
        x = F.relu(self.ra3_conv3(x))
        ra3_feat = self.ra3_conv4(x)
        print('ra3_feat', ra3_feat.shape)
        x = ra3_feat + crop_3 + d4
        lateral_map_3 = F.interpolate(x, scale_factor=16, mode='bilinear')

        crop_2 = F.interpolate(x, scale_factor=2, mode='bilinear')
        x = -1 * (torch.sigmoid(crop_2)) + 1
        x = x.expand(-1, 512, -1, -1).mul(x2)
        x = self.ra2_conv1(x)
        x = F.relu(self.ra2_conv2(x))
        x = F.relu(self.ra2_conv3(x))
        ra2_feat = self.ra2_conv4(x)
        print('ra2_feat', ra2_feat.shape)
        print('d3', d3.shape)
        x = ra2_feat + crop_2 + d3
        lateral_map_2 = F.interpolate(x, scale_factor=8, mode='bilinear')

        return [lateral_map_5, lateral_map_4, lateral_map_3, lateral_map_2]


if __name__ == '__main__':
    # Automatically uses GPU if available, otherwise CPU
    images = torch.rand(1, 3, 384, 384).to(DEVICE)
    model = Pra_FATNet(num_classes=1, input_channels=3, deep_supervision=True)
    model = model.to(DEVICE)

    outputs = model(images)
    print(outputs[0].shape)
    print(outputs[1].shape)
    print(outputs[2].shape)
    print(outputs[3].shape)

[INFO] Using device: cuda


Downloading: "https://shanghuagao.oss-cn-beijing.aliyuncs.com/res2net/res2net50_v1b_26w_4s-3cf99910.pth" to /root/.cache/torch/hub/checkpoints/res2net50_v1b_26w_4s-3cf99910.pth
100%|██████████| 98.4M/98.4M [00:12<00:00, 8.40MB/s]
Downloading: "https://github.com/facebookresearch/deit/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/deit/deit_base_distilled_patch16_384-d0272ac0.pth" to /root/.cache/torch/hub/checkpoints/deit_base_distilled_patch16_384-d0272ac0.pth
100%|██████████| 334M/334M [00:01<00:00, 196MB/s]  


x torch.Size([1, 64, 96, 96])
x1 torch.Size([1, 256, 96, 96])
x2 torch.Size([1, 512, 48, 48])
x3 torch.Size([1, 1024, 24, 24])
x4 torch.Size([1, 2048, 12, 12])
emb torch.Size([1, 576, 768])
e3 torch.Size([1, 1024, 24, 24])
e2 torch.Size([1, 512, 48, 48])
e1 torch.Size([1, 256, 96, 96])
feature_out torch.Size([1, 2048, 12, 12])
d4 torch.Size([1, 1024, 24, 24])
d3 torch.Size([1, 512, 48, 48])
d2 torch.Size([1, 256, 96, 96])
ra5_feat torch.Size([1, 1, 48, 48])
ra4_feat torch.Size([1, 1, 12, 12])
ra3_feat torch.Size([1, 1, 24, 24])
ra2_feat torch.Size([1, 1, 48, 48])
d3 torch.Size([1, 1, 48, 48])
torch.Size([1, 1, 384, 384])
torch.Size([1, 1, 384, 384])
torch.Size([1, 1, 384, 384])
torch.Size([1, 1, 384, 384])


In [3]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import cv2

# ---- Import your model ----
import sys
sys.path.append('/kaggle/input/models/anushka1511/pra-fatnet-local-py/pytorch/default/1/Pra_FATNet_local.py')
from res2net_v1b import res2net50_v1b_26w_4s

# ==============================================================
# CONFIG
# ==============================================================
class CFG:
    csv_path    = "/kaggle/input/isic-2017/train_data.csv"   # <-- update this
    img_size    = 384
    batch_size  = 4
    num_epochs  = 10
    lr          = 1e-4
    val_split   = 0.15
    num_workers = 2
    save_dir    = "/kaggle/working/checkpoints"
    device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed        = 42

os.makedirs(CFG.save_dir, exist_ok=True)
print(f"[INFO] Using device: {CFG.device}")

# ==============================================================
# DATASET
# ==============================================================
class SegDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load image (RGB)
        image = cv2.imread(row["image_path"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Load mask (grayscale) — binarize just in case
        mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)   # ensure 0.0 / 1.0

        if self.transforms:
            augmented = self.transforms(image=image, mask=mask)
            image = augmented["image"]
            mask  = augmented["mask"]

        # mask shape: (H, W) -> (1, H, W)
        mask = mask.unsqueeze(0)
        return image, mask


def get_transforms(img_size, mode="train"):
    if mode == "train":
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ColorJitter(brightness=0.2, contrast=0.2, p=0.3),
            A.Normalize(mean=(0.485, 0.456, 0.406),
                        std =(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=(0.485, 0.456, 0.406),
                        std =(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])


# ==============================================================
# LOSS  (BCE + Dice combined — standard for binary segmentation)
# ==============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred   = torch.sigmoid(pred)
        pred   = pred.view(-1)
        target = target.view(-1)
        intersection = (pred * target).sum()
        dice = (2. * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth)
        return 1 - dice


class BCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce  = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, pred, target):
        return self.bce(pred, target) + self.dice(pred, target)


# ==============================================================
# METRICS
# ==============================================================
def dice_score(pred, target, threshold=0.5, smooth=1.0):
    pred   = (torch.sigmoid(pred) > threshold).float()
    pred   = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return ((2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)).item()

def iou_score(pred, target, threshold=0.5, smooth=1.0):
    pred   = (torch.sigmoid(pred) > threshold).float()
    pred   = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return ((intersection + smooth) / (union + smooth)).item()


# ==============================================================
# TRAIN / VALIDATE ONE EPOCH
# ==============================================================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for images, masks in tqdm(loader, desc="  Train", leave=False):
        images = images.to(device)
        masks  = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)   # returns list of 4 maps

        # Compute loss on all 4 deep-supervision outputs
        # Resize each output to mask size before computing loss
        loss = 0
        for out in outputs:
            out_resized = F.interpolate(out, size=masks.shape[2:], mode="bilinear", align_corners=False)
            loss += criterion(out_resized, masks)
        loss = loss / len(outputs)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, total_dice, total_iou = 0.0, 0.0, 0.0

    for images, masks in tqdm(loader, desc="  Val  ", leave=False):
        images = images.to(device)
        masks  = masks.to(device)

        outputs = model(images)

        loss = 0
        for out in outputs:
            out_resized = F.interpolate(out, size=masks.shape[2:], mode="bilinear", align_corners=False)
            loss += criterion(out_resized, masks)
        loss = loss / len(outputs)

        # Use the final (highest-res) output for metrics
        final_out = F.interpolate(outputs[-1], size=masks.shape[2:], mode="bilinear", align_corners=False)
        total_loss += loss.item()
        total_dice += dice_score(final_out, masks)
        total_iou  += iou_score(final_out, masks)

    n = len(loader)
    return total_loss / n, total_dice / n, total_iou / n


# ==============================================================
# MAIN
# ==============================================================
def main():
    torch.manual_seed(CFG.seed)

    # ---- Load CSV ----
    df = pd.read_csv(CFG.csv_path)
    print(f"[INFO] Total samples: {len(df)}")

    # Optional: filter only positive samples (label == 1)
    # df = df[df["label"] == 1].reset_index(drop=True)

    train_df, val_df = train_test_split(
        df, test_size=CFG.val_split, random_state=CFG.seed, stratify=df["label"]
    )
    print(f"[INFO] Train: {len(train_df)}  |  Val: {len(val_df)}")

    train_ds = SegDataset(train_df, transforms=get_transforms(CFG.img_size, "train"))
    val_ds   = SegDataset(val_df,   transforms=get_transforms(CFG.img_size, "val"))

    train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                              num_workers=CFG.num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CFG.batch_size, shuffle=False,
                              num_workers=CFG.num_workers, pin_memory=True)

    # ---- Model ----
    model = Pra_FATNet(num_classes=1, input_channels=3, deep_supervision=True)
    model = model.to(CFG.device)

    # ---- Optimizer & Scheduler ----
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.num_epochs, eta_min=1e-6)
    criterion = BCEDiceLoss()

    # ---- Training Loop ----
    best_dice = 0.0

    for epoch in range(1, CFG.num_epochs + 1):
        print(f"\nEpoch [{epoch}/{CFG.num_epochs}]  lr={scheduler.get_last_lr()[0]:.2e}")

        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, CFG.device)
        val_loss, val_dice, val_iou = validate(model, val_loader, criterion, CFG.device)

        scheduler.step()

        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val   Loss: {val_loss:.4f}  |  Dice: {val_dice:.4f}  |  IoU: {val_iou:.4f}")

        # Save best model
        if val_dice > best_dice:
            best_dice = val_dice
            ckpt_path = os.path.join(CFG.save_dir, "best_model.pth")
            torch.save(model.state_dict(), ckpt_path)
            print(f"  ✅ Saved best model  (Dice={best_dice:.4f})  →  {ckpt_path}")

    print(f"\n[DONE] Best Val Dice: {best_dice:.4f}")


if __name__ == "__main__":
    main()

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


[INFO] Using device: cuda
[INFO] Total samples: 1790
[INFO] Train: 1521  |  Val: 269


Using cache found in /root/.cache/torch/hub/facebookresearch_deit_main



Epoch [1/10]  lr=1.00e-04


  Train:   0%|          | 0/381 [00:00<?, ?it/s]

...

e3 torch.Size([4, 1024, 24, 24])
e2 torch.Size([4, 512, 48, 48])
e1 torch.Size([4, 256, 96, 96])
feature_out torch.Size([4, 2048, 12, 12])
d4 torch.Size([4, 1024, 24, 24])
d3 torch.Size([4, 512, 48, 48])
d2 torch.Size([4, 256, 96, 96])
ra5_feat torch.Size([4, 1, 48, 48])
ra4_feat torch.Size([4, 1, 12, 12])
ra3_feat torch.Size([4, 1, 24, 24])
ra2_feat torch.Size([4, 1, 48, 48])
d3 torch.Size([4, 1, 48, 48])
x torch.Size([4, 64, 96, 96])
x1 torch.Size([4, 256, 96, 96])
x2 torch.Size([4, 512, 48, 48])
x3 torch.Size([4, 1024, 24, 24])
x4 torch.Size([4, 2048, 12, 12])
emb torch.Size([4, 576, 768])


  Val  :  99%|█████████▊| 67/68 [00:37<00:00,  1.84it/s]

e3 torch.Size([4, 1024, 24, 24])
e2 torch.Size([4, 512, 48, 48])
e1 torch.Size([4, 256, 96, 96])
feature_out torch.Size([4, 2048, 12, 12])
d4 torch.Size([4, 1024, 24, 24])
d3 torch.Size([4, 512, 48, 48])
d2 torch.Size([4, 256, 96, 96])
ra5_feat torch.Size([4, 1, 48, 48])
ra4_feat torch.Size([4, 1, 12, 12])
ra3_feat torch.Size([4, 1, 24, 24])
ra2_feat torch.Size([4, 1, 48, 48])
d3 torch.Size([4, 1, 48, 48])
x torch.Size([1, 64, 96, 96])
x1 torch.Size([1, 256, 96, 96])
x2 torch.Size([1, 512, 48, 48])
x3 torch.Size([1, 1024, 24, 24])
x4 torch.Size([1, 2048, 12, 12])
emb torch.Size([1, 576, 768])


e3 torch.Size([1, 1024, 24, 24])
e2 torch.Size([1, 512, 48, 48])
e1 torch.Size([1, 256, 96, 96])
feature_out torch.Size([1, 2048, 12, 12])
d4 torch.Size([1, 1024, 24, 24])
d3 torch.Size([1, 512, 48, 48])
d2 torch.Size([1, 256, 96, 96])
ra5_feat torch.Size([1, 1, 48, 48])
ra4_feat torch.Size([1, 1, 12, 12])
ra3_feat torch.Size([1, 1, 24, 24])
ra2_feat torch.Size([1, 1, 48, 48])
d3 torch.Size([1, 1, 48, 48])
  Train Loss: 0.1740
  Val   Loss: 0.1883  |  Dice: 0.9132  |  IoU: 0.8431


  ✅ Saved best model  (Dice=0.9132)  →  /kaggle/working/checkpoints/best_model.pth

[DONE] Best Val Dice: 0.9132


In [4]:
val_loss, val_dice, val_iou = validate(model, val_loader, criterion, CFG.device)

# ADD THIS:
print(f"  Val   Loss: {val_loss:.4f}  |  Dice: {val_dice:.4f}  |  IoU: {val_iou:.4f}")

NameError: name 'val_loader' is not defined